# 🌐 Web Research Agent: A Fixed Pipeline vs. a ReAct Loop

## Learning Objectives
In this notebook, you will learn:
1. **A research pipeline in LangGraph** - Plan questions → search the web for each → compile a cited report
2. **Custom state and reducers** - State fields beyond `messages`, and a reducer that merges search results
3. **Planning, acting and reporting nodes** - How each node reads and updates only its part of the state
4. **The same task as a ReAct agent** - Letting the model decide what to search next, and when to stop
5. **Choosing between them** - Predictability and cost vs. adaptivity

## Where this fits
`03_react_with_create_agent.ipynb` packaged the ReAct loop with `create_agent`. Research is the classic ReAct application — but the most common "research agent" design is not a loop at all: it plans all its searches up front and runs them in a straight line. This notebook builds that pipeline first, then gives the same topic to a ReAct agent so you can see what the loop changes. Next, `05_swe_agent_applied.ipynb` applies the loop to fixing code.

## Prerequisites
- LangGraph state graphs — `01_react_agent_langgraph.ipynb`; `create_agent` — `03_react_with_create_agent.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env` (live web search)

---
## 📦 Step 1: Environment Setup

Load the API keys, create the LLM, and a Tavily client for web search. The research size is kept small (3 questions × 3 results) so a full run costs a few cents.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


In [2]:
# ============================================================================
# LLM AND SEARCH CLIENT INITIALIZATION
# ============================================================================
from tavily import TavilyClient

from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# Research size: how many questions to plan, and how many results per search
NUM_QUESTIONS = 3
MAX_SEARCH_RESULTS = 3

print(f"✅ LLM initialized: {llm.model_name}")
print(f"✅ Tavily client ready ({NUM_QUESTIONS} questions × {MAX_SEARCH_RESULTS} results)")

✅ LLM initialized: gpt-4o-mini
✅ Tavily client ready (3 questions × 3 results)


---
## 🗂️ Part 1: The Research State

The pipeline passes more than messages between nodes: the topic, the planned questions, the search results per question, and the final report. Each node returns **only the keys it changes**; LangGraph merges them into the state.

### Key Concepts:
- **Reducer**: a function that tells LangGraph how to combine a node's update with the existing value. `merge_search_results` merges dictionaries; `operator.add` appends to the `log` list
- **Partial updates**: returning `{"research_questions": [...]}` updates that key and leaves the rest alone

In [3]:
# ============================================================================
# STATE DEFINITION: Topic, questions, results, report and a log
# ============================================================================
import operator
from typing import Annotated, Dict, List, TypedDict


def merge_search_results(old: Dict[str, List[Dict]] | None,
                         new: Dict[str, List[Dict]] | None) -> Dict[str, List[Dict]]:
    """Reducer: merge dicts of {question: results} instead of overwriting."""
    merged = dict(old or {})
    merged.update(new or {})
    return merged


class ResearchState(TypedDict):
    """
    topic: The research topic provided by the user
    research_questions: Questions generated by the LLM in the planning node
    search_results: {question: [search hits]} filled by the acting node
    final_report: The compiled research report
    log: One entry per node, appended (operator.add) — a trace of the run
    """

    topic: str
    research_questions: List[str]
    search_results: Annotated[Dict[str, List[Dict]], merge_search_results]
    final_report: str
    log: Annotated[List[str], operator.add]


print(f"✅ ResearchState keys: {list(ResearchState.__annotations__)}")

✅ ResearchState keys: ['topic', 'research_questions', 'search_results', 'final_report', 'log']


---
## 🧠 Part 2: The Planning Node

The planning node asks the LLM for a fixed number of research questions and parses them out of a numbered list. This is where the pipeline commits to its whole plan — before it has seen a single search result.

In [4]:
# ============================================================================
# PLANNING NODE: Generate research questions with the LLM
# ============================================================================
import re

from langchain_core.messages import HumanMessage, SystemMessage


def parse_questions(text: str, limit: int) -> List[str]:
    """Strip numbering ("1.", "1)", "Question 1:") and keep non-trivial lines."""
    questions = []
    for line in text.strip().split("\n"):
        cleaned = re.sub(r"^\s*\d+[\.\)\:]?\s*", "", line.strip())
        cleaned = re.sub(r"^question\s+\d+[\.\)\:]?\s*", "", cleaned, flags=re.IGNORECASE)
        if len(cleaned) > 10:  # filter out very short lines
            questions.append(cleaned)
    return questions[:limit]


def planning_node(state: ResearchState) -> dict:
    """Reason: turn the topic into NUM_QUESTIONS searchable questions."""
    system_prompt = f"""You are an expert research assistant. Your task is to generate
    comprehensive research questions for a given topic. Generate questions that:
    1. Cover different aspects of the topic (causes, effects, solutions, trends)
    2. Are specific and answerable through web search
    3. Progress from fundamental to advanced understanding
    4. Include current developments and future implications

    Generate EXACTLY {NUM_QUESTIONS} well-structured research questions."""
    user_prompt = (
        f"Topic: {state['topic']}\n\n"
        f"Please generate {NUM_QUESTIONS} research questions about this topic.\n"
        "Format your response as a numbered list with ONLY the questions, one per line."
    )
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
    questions = parse_questions(response.content, NUM_QUESTIONS)

    print(f"🧠 PLANNING: {len(questions)} questions for '{state['topic']}'")
    for i, q in enumerate(questions, 1):
        print(f"   {i}. {q}")
    return {"research_questions": questions, "log": [f"planning: {len(questions)} questions"]}

---
## 🔍 Part 3: The Acting and Reporting Nodes

The **acting** node runs one web search per planned question — no matter what the earlier searches returned. The **reporting** node hands every result to the LLM and asks for a structured, cited report.

In [5]:
# ============================================================================
# ACTING NODE: One web search per planned question
# ============================================================================
def acting_node(state: ResearchState) -> dict:
    """Act: search the web for every research question."""
    search_results = {}
    print("🔍 ACTING: web search")
    for i, question in enumerate(state["research_questions"], 1):
        try:
            response = tavily_client.search(query=question, max_results=MAX_SEARCH_RESULTS)
            search_results[question] = [
                {
                    "title": r.get("title", "No title"),
                    "url": r.get("url", ""),
                    "content": r.get("content", "No content available"),
                }
                for r in response.get("results", [])
            ]
            print(f"   ✓ Q{i}: {len(search_results[question])} sources")
        except Exception as e:
            # A failed search is recorded, not fatal: the report notes the gap
            print(f"   ✗ Q{i}: search failed ({e})")
            search_results[question] = []
    total = sum(len(v) for v in search_results.values())
    return {"search_results": search_results, "log": [f"acting: {total} sources"]}

In [6]:
# ============================================================================
# REPORTING NODE: Compile a cited report from all results
# ============================================================================
def format_research_data(questions: List[str], search_results: Dict[str, List[Dict]]) -> str:
    """Lay out every question with its sources for the LLM."""
    lines = []
    for i, question in enumerate(questions, 1):
        lines.append(f"\n### Question {i}: {question}\n")
        results = search_results.get(question, [])
        if not results:
            lines.append("No search results found.\n")
        for j, r in enumerate(results, 1):
            lines.append(f"{j}. **{r['title']}**\n   URL: {r['url']}\n   Content: {r['content'][:500]}...\n")
    return "\n".join(lines)


def reporting_node(state: ResearchState) -> dict:
    """Synthesize the findings into a structured report with citations."""
    system_prompt = """You are an expert research writer. Compile a well-structured research
    report from the gathered information. Synthesize across sources, cite them in
    [Source Title](https://github.com/Sourav692/ai-engineering-demystified-06-agent-fundamentals/tree/main/modules/03_react_agents/notebooks/URL) format, and write a short introduction and conclusion."""
    research_data = format_research_data(state["research_questions"], state["search_results"])
    user_prompt = f"""Topic: {state['topic']}

Research Data:
{research_data}

Write the report with these sections: Introduction; Research Findings (one subsection per
question, with citations); Conclusion; References. Keep it under 700 words."""
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
    print(f"📝 REPORTING: {len(response.content)} characters")
    return {"final_report": response.content, "log": ["reporting: report compiled"]}

---
## 🧵 Part 4: Wiring and Running the Pipeline

The graph is a straight line — `planning → acting → reporting → END`. No edge ever goes back: the pipeline cannot notice that a search came back empty, or that an answer raises a new question.

In [7]:
# ============================================================================
# GRAPH CONSTRUCTION: START → planning → acting → reporting → END
# ============================================================================
from langgraph.graph import END, START, StateGraph

workflow = StateGraph(ResearchState)
workflow.add_node("planning", planning_node)
workflow.add_node("acting", acting_node)
workflow.add_node("reporting", reporting_node)

workflow.add_edge(START, "planning")
workflow.add_edge("planning", "acting")
workflow.add_edge("acting", "reporting")
workflow.add_edge("reporting", END)

research_pipeline = workflow.compile()

print("✅ Pipeline compiled:", " → ".join(n for n in research_pipeline.get_graph().nodes))

✅ Pipeline compiled: __start__ → planning → acting → reporting → __end__


In [8]:
# ============================================================================
# RUN THE PIPELINE
# ============================================================================
from IPython.display import Markdown, display

TOPIC = "Artificial Intelligence in Healthcare"

pipeline_state = research_pipeline.invoke({
    "topic": TOPIC,
    "research_questions": [],
    "search_results": {},
    "final_report": "",
    "log": [],
})

print("\n🧾 Run log:", pipeline_state["log"])
display(Markdown(pipeline_state["final_report"]))

🧠 PLANNING: 3 questions for 'Artificial Intelligence in Healthcare'
   1. What are the primary applications of artificial intelligence in healthcare, and how do they improve patient outcomes and operational efficiency in medical settings?
   2. What ethical concerns and challenges arise from the implementation of artificial intelligence in healthcare, particularly regarding data privacy and algorithmic bias?
   3. How is the integration of artificial intelligence in healthcare expected to evolve over the next decade, and what potential advancements could significantly transform patient care and medical research?
🔍 ACTING: web search
   ✓ Q1: 3 sources
   ✓ Q2: 3 sources
   ✓ Q3: 3 sources
📝 REPORTING: 7229 characters

🧾 Run log: ['planning: 3 questions', 'acting: 9 sources', 'reporting: report compiled']


# Artificial Intelligence in Healthcare: A Comprehensive Research Report

## Introduction

Artificial Intelligence (AI) is rapidly transforming the healthcare landscape, offering innovative solutions that enhance patient outcomes and operational efficiency. As healthcare systems grapple with increasing demands and complexities, AI technologies are emerging as pivotal tools in clinical settings. This report synthesizes current research findings on the applications of AI in healthcare, the ethical challenges associated with its implementation, and the anticipated evolution of AI technologies over the next decade.

## Research Findings

### 1. Primary Applications of Artificial Intelligence in Healthcare

AI applications in healthcare are diverse and impactful, significantly improving patient outcomes and operational efficiency. Key areas of application include:

- **Clinical Decision Support**: AI algorithms analyze vast amounts of data to assist healthcare professionals in making informed decisions, leading to improved diagnostic accuracy and treatment plans ([Simbo AI - Blogs](https://www.simbo.ai/blog/exploring-the-role-of-artificial-intelligence-in-enhancing-patient-care-outcomes-and-operational-efficiency-in-modern-healthcare-settings-288081)).
  
- **Patient Monitoring and Management**: AI technologies enable continuous monitoring of patients through wearable devices, allowing for timely interventions and personalized care ([Simbo AI - Blogs](https://www.simbo.ai/blog/the-impact-of-artificial-intelligence-on-enhancing-patient-outcomes-and-operational-efficiency-in-healthcare-settings-1734608)).
  
- **Administrative Efficiency**: AI streamlines administrative tasks such as scheduling, billing, and documentation, freeing up healthcare professionals to focus on patient care. For instance, ambient AI scribe technologies automate documentation processes, enhancing workflow efficiency ([How to Improve Patient Outcomes with AI Technology in Healthcare](https://examples.tely.ai/how-to-improve-patient-outcomes-with-ai-technology-in-healthcare)).

These applications collectively contribute to a more efficient healthcare system, ultimately leading to better patient outcomes and enhanced operational performance.

### 2. Ethical Concerns and Challenges in AI Implementation

The integration of AI in healthcare raises significant ethical concerns, particularly regarding data privacy and algorithmic bias. Key issues include:

- **Data Privacy**: AI systems rely on sensitive health data, raising concerns about patient confidentiality and compliance with regulations such as HIPAA. Despite existing frameworks, challenges persist in safeguarding patient information ([Ethical concerns and strategies for implementing artificial intelligence in healthcare](https://link.springer.com/article/10.1186/s12910-026-01404-8)).
  
- **Algorithmic Bias**: AI algorithms can perpetuate existing biases in healthcare data, leading to disparities in treatment outcomes. This necessitates the establishment of transparent and accountable AI systems to ensure equitable healthcare delivery ([The ethics of using artificial intelligence in medical research](https://www.kosinmedj.org/journal/view.php?doi=10.7180%2Fkmj.24.140)).
  
- **Informed Consent and Accountability**: The deployment of AI in clinical decision-making raises questions about informed consent and the accountability of AI systems. A clear ethical framework is essential to enhance transparency and maintain patient trust ([Ethics of AI in Healthcare: Addressing Privacy, Bias & | Alation](https://www.alation.com/blog/ethics-of-ai-in-healthcare-privacy-bias-trust-2025)).

### 3. Future Evolution of AI in Healthcare

The next decade is expected to witness significant advancements in AI technologies, fundamentally reshaping patient care and medical research. Anticipated developments include:

- **Personalized Medicine**: AI will enable more tailored treatment plans based on individual patient data, enhancing the effectiveness of interventions ([The Future of AI in Healthcare: Big Shifts to Expect in the Next Decade](https://www.wokelo.ai/blog/future-of-ai-in-healthcare-what-to-expect-by-2035)).
  
- **Enhanced Diagnostics**: AI's ability to analyze complex datasets will lead to breakthroughs in diagnostic accuracy, particularly in areas such as imaging and pathology ([How AI Agents and Tech Will Transform Healthcare | BCG](https://www.bcg.com/publications/2026/how-ai-agents-will-transform-health-care)).
  
- **Integration into Daily Workflows**: As AI becomes more embedded in healthcare practices, it will shift from merely supporting tasks to making autonomous decisions, thereby transforming care delivery models and workforce dynamics ([How AI Will Shape the Future of Health Care In 2026 - SullivanCotter](https://sullivancotter.com/resources/how-ai-will-shape-the-future-of-health-care-in-2026)).

## Conclusion

Artificial Intelligence is poised to revolutionize healthcare by enhancing patient outcomes and operational efficiency. However, the ethical implications of its implementation must be carefully navigated to ensure patient trust and equity in care. As AI technologies continue to evolve, their integration into healthcare systems will require ongoing attention to ethical standards and regulatory frameworks. The future of healthcare, driven by AI, promises to be more personalized, efficient, and effective, ultimately benefiting patients and providers alike.

## References

1. Simbo AI - Blogs. [Exploring the Role of Artificial Intelligence in Enhancing Patient Care Outcomes and Operational Efficiency in Modern Healthcare Settings](https://www.simbo.ai/blog/exploring-the-role-of-artificial-intelligence-in-enhancing-patient-care-outcomes-and-operational-efficiency-in-modern-healthcare-settings-288081).
2. Simbo AI - Blogs. [The Impact of Artificial Intelligence on Enhancing Patient Outcomes and Operational Efficiency in Healthcare Settings](https://www.simbo.ai/blog/the-impact-of-artificial-intelligence-on-enhancing-patient-outcomes-and-operational-efficiency-in-healthcare-settings-1734608).
3. [How to Improve Patient Outcomes with AI Technology in Healthcare](https://examples.tely.ai/how-to-improve-patient-outcomes-with-ai-technology-in-healthcare).
4. [Ethical concerns and strategies for implementing artificial intelligence in healthcare: a review of empirical studies](https://link.springer.com/article/10.1186/s12910-026-01404-8).
5. [The ethics of using artificial intelligence in medical research](https://www.kosinmedj.org/journal/view.php?doi=10.7180%2Fkmj.24.140).
6. [Ethics of AI in Healthcare: Addressing Privacy, Bias & | Alation](https://www.alation.com/blog/ethics-of-ai-in-healthcare-privacy-bias-trust-2025).
7. [The Future of AI in Healthcare: Big Shifts to Expect in the Next Decade](https://www.wokelo.ai/blog/future-of-ai-in-healthcare-what-to-expect-by-2035).
8. [How AI Agents and Tech Will Transform Healthcare | BCG](https://www.bcg.com/publications/2026/how-ai-agents-will-transform-health-care).
9. [How AI Will Shape the Future of Health Care In 2026 - SullivanCotter](https://sullivancotter.com/resources/how-ai-will-shape-the-future-of-health-care-in-2026).

The log shows exactly three steps, every time: the plan fixes the number of searches before any result is seen. That is the pipeline's strength — predictable cost, easy to test — and its limit.

---
## 🔁 Part 5: The Same Research as a ReAct Agent

Now give the same topic to a ReAct agent with one tool, `web_search`. There is no plan node and no fixed question count: after every search the model reads the results and decides whether to search again (and for what) or to write the report. A call cap bounds the loop.

In [9]:
# ============================================================================
# REACT RESEARCHER: create_agent with a web_search tool
# ============================================================================
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_core.tools import tool


@tool
def web_search(query: str) -> list[dict]:
    """Search the web. Returns title, URL and a content snippet for the top results."""
    print(f"   🔍 web_search({query!r})")
    response = tavily_client.search(query=query, max_results=MAX_SEARCH_RESULTS)
    return [{"title": r["title"], "url": r["url"], "content": r["content"][:500]}
            for r in response.get("results", [])]


react_researcher = create_agent(
    llm,
    [web_search],
    system_prompt=(
        "You are an expert research assistant. Research the user's topic with web_search, "
        "ONE search per turn. Start broad. After each result, pick the most important gap "
        "or the most promising specific lead it mentions (a named system, a risk, a number) "
        "and make your next search about that. Stop after 3 to 5 searches, when you can "
        "cover the topic's main aspects. Then write a report (under 700 words) with an "
        "Introduction, Findings, Conclusion and References, citing sources as [Source Title](https://github.com/Sourav692/ai-engineering-demystified-06-agent-fundamentals/tree/main/modules/03_react_agents/notebooks/URL)."
    ),
    # Bound the loop: at most 8 model calls per run
    middleware=[ModelCallLimitMiddleware(run_limit=8, exit_behavior="end")],
)

print("🔁 ReAct researcher at work:")
react_state = react_researcher.invoke({"messages": [("user", f"Research topic: {TOPIC}")]})

react_searches = [tc["args"]["query"] for m in react_state["messages"]
                  for tc in getattr(m, "tool_calls", [])]
display(Markdown(react_state["messages"][-1].content))

🔁 ReAct researcher at work:
   🔍 web_search('Artificial Intelligence in Healthcare 2023')
   🔍 web_search('AI models in surgery Duke University 2023')
   🔍 web_search('AI in healthcare physician adoption statistics 2023')
   🔍 web_search('AI in healthcare risks and challenges 2023')


# Report on Artificial Intelligence in Healthcare

## Introduction
Artificial Intelligence (AI) is increasingly transforming the healthcare landscape, offering innovative solutions to improve patient care, streamline operations, and enhance decision-making processes. This report explores the current state of AI in healthcare, focusing on its applications, adoption rates among healthcare professionals, and the associated risks and challenges.

## Findings

### Applications of AI in Healthcare
AI technologies are being utilized across various medical specialties, particularly in imaging, surgery, and oncology. A notable example is the implementation of AI models at Duke University Health System, which demonstrated a 13% increase in accuracy for predicting operating room times compared to human schedulers. These models were trained on extensive datasets from thousands of surgical cases and are now actively used in all operating rooms at Duke, showcasing the practical benefits of AI in enhancing surgical efficiency and patient management [Artificial Intelligence in Health Care: Promise and Pitfalls](https://medschool.duke.edu/news/artificial-intelligence-health-care-promise-and-pitfalls).

### Adoption Rates Among Physicians
The adoption of AI technologies by healthcare professionals has seen a significant increase. A survey conducted by the American Medical Association revealed that 66% of physicians reported using AI in their practices in 2023, marking a 78% increase from the previous year. This trend indicates a growing acceptance of AI tools, with physicians increasingly relying on AI for tasks such as summarizing medical research and generating care plans [2 in 3 physicians are using health AI—up 78% from 2023](https://www.ama-assn.org/practice-management/digital-health/2-3-physicians-are-using-health-ai-78-2023).

### Risks and Challenges
Despite the promising advancements, the integration of AI in healthcare is not without its challenges. Key risks include potential errors in AI systems that could lead to patient harm, concerns regarding patient privacy due to data acquisition, and the need for transparency in AI decision-making processes. A report from Brookings highlights these risks and emphasizes the importance of developing robust frameworks to mitigate them [Risks and remedies for artificial intelligence in health care](https://www.brookings.edu/articles/risks-and-remedies-for-artificial-intelligence-in-health-care).

### Future Trends
Looking ahead, the landscape of AI in healthcare is expected to evolve further. By 2026, it is projected that 81% of physicians will be using AI, up from 38% in 2023. This rapid growth underscores the increasing reliance on AI technologies to enhance clinical workflows and patient outcomes. However, it also raises concerns about over-reliance on AI, with many physicians expressing worries about potential skill degradation and the need for their involvement in AI adoption processes [100+ Latest AI in Healthcare Statistics Every Healthcare Leader Should Know](https://www.dialoghealth.com/post/ai-healthcare-statistics).

## Conclusion
Artificial Intelligence is poised to revolutionize healthcare by improving efficiency, accuracy, and patient care. The significant increase in adoption rates among physicians indicates a positive trend towards integrating AI into clinical practice. However, it is crucial to address the associated risks and challenges to ensure that AI technologies are implemented safely and effectively. Ongoing research, ethical considerations, and robust regulatory frameworks will be essential in navigating the future of AI in healthcare.

## References
1. [Artificial Intelligence in Health Care: Promise and Pitfalls](https://medschool.duke.edu/news/artificial-intelligence-health-care-promise-and-pitfalls)
2. [2 in 3 physicians are using health AI—up 78% from 2023](https://www.ama-assn.org/practice-management/digital-health/2-3-physicians-are-using-health-ai-78-2023)
3. [Risks and remedies for artificial intelligence in health care](https://www.brookings.edu/articles/risks-and-remedies-for-artificial-intelligence-in-health-care)
4. [100+ Latest AI in Healthcare Statistics Every Healthcare Leader Should Know](https://www.dialoghealth.com/post/ai-healthcare-statistics)

---
## ⚖️ Part 6: Comparing the Two

In [10]:
# ============================================================================
# COMPARE: searches, model calls and report size
# ============================================================================
pipeline_searches = pipeline_state["research_questions"]
react_model_calls = sum(1 for m in react_state["messages"] if m.type == "ai")

print(f"{'':<22}{'Pipeline':>12}{'ReAct':>12}")
print(f"{'Web searches':<22}{len(pipeline_searches):>12}{len(react_searches):>12}")
print(f"{'LLM calls':<22}{2:>12}{react_model_calls:>12}")
print(f"{'Report characters':<22}{len(pipeline_state['final_report']):>12}"
      f"{len(react_state['messages'][-1].content):>12}")

print("\nPipeline searched (planned up front):")
for q in pipeline_searches:
    print(f"   - {q}")
print("ReAct searched (decided one at a time):")
for q in react_searches:
    print(f"   - {q}")

                          Pipeline       ReAct
Web searches                     3           4
LLM calls                        2           5
Report characters             7229        4298

Pipeline searched (planned up front):
   - What are the primary applications of artificial intelligence in healthcare, and how do they improve patient outcomes and operational efficiency in medical settings?
   - What ethical concerns and challenges arise from the implementation of artificial intelligence in healthcare, particularly regarding data privacy and algorithmic bias?
   - How is the integration of artificial intelligence in healthcare expected to evolve over the next decade, and what potential advancements could significantly transform patient care and medical research?
ReAct searched (decided one at a time):
   - Artificial Intelligence in Healthcare 2023
   - AI models in surgery Duke University 2023
   - AI in healthcare physician adoption statistics 2023
   - AI in healthcare risks and 

Read the two search lists. The pipeline's questions were all written before any result came back, so they are broad and parallel. The ReAct agent's queries are chosen as it goes: it starts broad, then follows up on something a result mentioned (a named system, a statistic). Each later query should visibly build on an earlier result. Its search count is the model's decision (within the 3–5 the prompt asks for and the call cap enforces), which is both the point and the risk.

| | Fixed pipeline | ReAct agent |
|---|---|---|
| Who decides the searches | The plan, up front | The model, after each observation |
| Cost | Fixed and predictable | Varies; bound it with a call cap |
| Reacts to empty or surprising results | No | Yes |
| Easy to test and debug | Yes — same steps every run | Harder — the path changes run to run |
| Good for | Known-shape reports, batch jobs | Open questions, multi-hop facts |

---
## 📝 Summary

In this notebook, we learned:

### 1. A Research Pipeline in LangGraph
- `planning → acting → reporting` turns a topic into a cited report in three fixed steps

### 2. Custom State and Reducers
- State can hold any fields; reducers such as `operator.add` or a dict merge decide how updates combine
- Nodes return only the keys they change

### 3. Planning, Acting and Reporting Nodes
- The plan is fixed before any search result is seen; failed searches are recorded, not fatal

### 4. The Same Task as a ReAct Agent
- One `web_search` tool plus `create_agent` lets the model choose each next search from what it has read
- `ModelCallLimitMiddleware` bounds an open-ended research loop

### 5. Choosing Between Them
- Pipelines buy predictability; ReAct buys adaptivity. Many production researchers combine them: plan first, then let each step react

### Next Steps
- **`05_swe_agent_applied.ipynb`**: the ReAct loop applied to bug fixing, with file and test tools in a sandbox